In [ ]:
# ========================
# 0) 基本环境和依赖安装
# ========================
import os, sys, shutil, json, yaml
from tqdm import tqdm

base_dir = "/root/nfs/rank1_cot_exp"
if os.path.exists(base_dir):
    shutil.rmtree(base_dir)
os.makedirs(base_dir, exist_ok=True)
os.chdir(base_dir)

# 克隆依赖
os.system('git clone https://github.com/orionw/rank1.git .')
os.system('git clone https://github.com/hiyouga/LLaMA-Factory.git')
os.system('git lfs install')
os.system('git clone https://huggingface.co/datasets/jhu-clsp/rank1-training-data')
os.system('git clone --depth 1 https://huggingface.co/datasets/jhu-clsp/rank1-run-files -q')

# 安装 LLaMA-Factory
os.chdir('LLaMA-Factory')
os.system(f"{sys.executable} -m pip install .")
os.chdir('..')
print("\n *******Setup Complete******* ")

# 设置 HF_TOKEN
hf_token = "YOUR_HUGGINGFACE_TOKEN_HERE"
os.environ["HF_TOKEN"] = hf_token

# ✅ WandB
os.environ["WANDB_API_KEY"] = "YOUR_WANDB_API_KEY_HERE"  
os.environ["WANDB_PROJECT"] = "rank1_project"            # 可自定义项目名
os.environ["WANDB_MODE"] = "online"                      # 可选值: "online", "offline", "disabled"
# %%
# ========================
# 1) 生成 CoT 数据
# ========================
from datasets import load_dataset

dataset = load_dataset("./rank1-training-data", split='train')
output_file = "cot_data.jsonl"

with open(output_file, 'w', encoding='utf-8') as f:
    for item in tqdm(dataset, desc="正在处理"):
        try:
            input_text = item['input']
            query = input_text.split('Query:')[1].split('Passage:')[0].strip()
            passage = input_text.split('Passage:')[1].strip()
            output_text = item['output']
            if "<think>" in output_text and "</think>" in output_text:
                thought = output_text.split("<think>")[1].split("</think>")[0].strip()
                label = output_text.split("</think>")[-1].strip()
            else:
                continue
            if label not in ['true', 'false']:
                continue
            response = f"Reasoning: {thought}\nAnswer: {label}"
            new_sample = {"query": query, "passage": passage, "label": response}
            f.write(json.dumps(new_sample, ensure_ascii=False) + '\n')
        except (KeyError, IndexError):
            continue

print(f"\n✅ 'CoT版'数据创建完成: {output_file}")

# %%
# ========================
# 2) 生成 dataset_info.json
# ========================
os.makedirs('configs', exist_ok=True)

# 核心修正：将数据文件移动到 configs 目录
shutil.move(output_file, f"configs/{output_file}")
print(f"✅ 已将 {output_file} 移动到 configs/ 目录")


dataset_info = {
    "rank1_cot": {
        "file_name": "cot_data.jsonl", # 现在 LLaMA-Factory 可以在 `configs` 目录下找到它
        "columns": {
            "prompt": "query",
            "query": "passage",
            "response": "label"
        }
    }
}
dataset_info_path = 'configs/dataset_info.json'
with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)
print('✅ dataset_info.json 已创建')


# %%
# ========================
# 3) 生成训练配置 YAML
# ========================
train_config_3b = {
    'model_name_or_path': 'Qwen/Qwen2.5-3B',
    'output_dir': 'cot_lora_3b',
    'finetuning_type': 'lora',
    'lora_target': 'all',
    'per_device_train_batch_size': 8,
    'gradient_accumulation_steps': 2,
    'learning_rate': 0.0001,
    'num_train_epochs': 1,
    'lr_scheduler_type': 'cosine',
    'fp16': True,
    'do_train': True,
    'dataset': 'rank1_cot',
    'dataset_dir': 'configs',
    'template': 'default',
}
train_config_1_5b = train_config_3b.copy()
train_config_1_5b['model_name_or_path'] = 'Qwen/Qwen2.5-1.5B'
train_config_1_5b['output_dir'] = 'cot_lora_1_5b'

with open('configs/train_config_cot_3b.yaml', 'w', encoding='utf-8') as f:
    yaml.safe_dump(train_config_3b, f, allow_unicode=True)
with open('configs/train_config_cot_1_5b.yaml', 'w', encoding='utf-8') as f:
    yaml.safe_dump(train_config_1_5b, f, allow_unicode=True)
print('✅ 已写入 3B 与 1.5B 配置')

# %%
# ========================
# 4) 训练模型
# ========================
print("开始训练 3B ...")
!python LLaMA-Factory/src/train.py configs/train_config_cot_3b.yaml

print("开始训练 1.5B ...")
!python LLaMA-Factory/src/train.py configs/train_config_cot_1_5b.yaml

# %%
# ========================
# 5) LoRA 合并与上传
# ========================
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig
from huggingface_hub import HfApi
import torch, os, shutil

def merge_and_upload(base_model_path, local_lora_checkpoint_path, huggingface_repo_id):
    if not os.path.exists(local_lora_checkpoint_path):
        print(f"❌ LoRA checkpoint 未找到: {local_lora_checkpoint_path}")
        return

    print(f"开始合并 LoRA: {local_lora_checkpoint_path} → {huggingface_repo_id}")

    # 加载基础模型
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_path,
        torch_dtype=torch.float16,
        device_map="auto",
        use_auth_token=hf_token
    )

    tokenizer = AutoTokenizer.from_pretrained(
        base_model_path,
        use_auth_token=hf_token
    )

    # 确保本地有 adapter_config.json
    if not os.path.exists(os.path.join(local_lora_checkpoint_path, "adapter_config.json")):
        raise FileNotFoundError(f"⚠️ 缺少 adapter_config.json，请确认 {local_lora_checkpoint_path} 是否完整保存 LoRA")

    # 加载 LoRA 并合并
    peft_model = PeftModel.from_pretrained(base_model, local_lora_checkpoint_path)
    merged_model = peft_model.merge_and_unload()

    # 保存合并后的模型
    save_path = f"/root/nfs/{huggingface_repo_id.replace('/', '_')}_merged"
    shutil.rmtree(save_path, ignore_errors=True)
    merged_model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    # 上传 HuggingFace
    api = HfApi()
    api.create_repo(repo_id=huggingface_repo_id, exist_ok=True, token=hf_token)
    api.upload_folder(folder_path=save_path, repo_id=huggingface_repo_id, repo_type="model", token=hf_token)
    print(f"✅ 上传成功: {huggingface_repo_id}")

# 训练完成后再执行
merge_and_upload("Qwen/Qwen2.5-3B", "./cot_lora_3b", "YOUR_HF_USERNAME/rank1-cot-3b")
merge_and_upload("Qwen/Qwen2.5-1.5B", "./cot_lora_1_5b", "YOUR_HF_USERNAME/rank1-cot-1-5b")

